In [4]:
import os
import pandas as pd
from typing import Sequence


import ee
import geemap.foliumap as geemap
import geemap

from find_set_root import find_set_project_root
PROJECT_ROOT = find_set_project_root()
print(f"Project root found at: {PROJECT_ROOT}")
import utils.general_functions as ugf


DIR_RAW = os.path.join(PROJECT_ROOT, 'data', 'raw')
DIR_DERIVED = os.path.join(PROJECT_ROOT, 'data', 'derived')
ugf.dir_ensure([DIR_RAW, DIR_DERIVED])

GEE_PROJECT = 'ee-tymc5571-multi-disturbance'
DRIVE_FOLDER = "GEE_Exports_resilience_20251209"

#Prepare to use Earth Engine
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
)
ee.Initialize(project=GEE_PROJECT)

SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file


#############
# USER SET PARAMETERS
#############

TARGET_CRS = 'EPSG:5070'  # Albers Equal Area - selected since working in W.US
SCALE = 250  # in meters - approximate MODIS resolution
MINIMUM_FOREST_PERC_PIXEL = 0.50  # minimum percent of coarse forest at pixel to be included
MINIMUM_FIRE_PERC_PIXEL = 0.50  # minimum percent of pixel within fire perimeter to be included as a burnt pixel
MINIMUM_USFS_PERC_PIXEL = 0.95  # minimum percent of pixel within USFS to be included in data

# Setup aoi
AOI_REGION = '6.2.14'
AOI_READABLE = "SRockies"

# AOI_REGION = '6.2.15'
# AOI_READABLE = "IDBatholith"

# AOI_REGION = '6.2.12'
# AOI_READABLE = "SierraNevada"

# AOI_REGION = '6.2.5'
# AOI_READABLE = "NCascades"

#AOI_READABLE = "test"

COARSE_FOREST_PERC_MIN = 0
COARSE_TRE_MAX_MIN = 25

FOREST_RANDOM_SAMPLE = 0.2
RANDOM_SEED = 1234
TILE_SIZE = 100000 #100000 works for both 1000 and 500 sample scale
SAMPLE_SCALE = 500  # in meters

Project root found at: C:\Users\tymc5571\dev\compound-disturbance-resilience
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\raw
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\derived


In [2]:
epa_lvl3 = ee.FeatureCollection("EPA/Ecoregions/2013/L3")
epa_lvl4 = ee.FeatureCollection("EPA/Ecoregions/2013/L4")
wumi_fires = ee.FeatureCollection("projects/ee-tymc5571-cbi-module/assets/WUMI2024a_main_fires_unified_simple_no_circles")


# -------------------------------------------------------------------
# Get Colorado boundary
# -------------------------------------------------------------------
states = ee.FeatureCollection("TIGER/2018/States")
colorado = states.filter(ee.Filter.eq("NAME", "Colorado")).geometry()

# -------------------------------------------------------------------
# Generate N random points inside Colorado
# -------------------------------------------------------------------
N = 1000          # number of points
SEED = 42         # for reproducibility

points = ee.FeatureCollection.randomPoints(
    region=colorado,
    points=N,
    seed=SEED,
    maxError=1
)

# Optional: add a point_id
points = points.map(lambda f: f.set("point_id", f.id()))



In [17]:
m = geemap.Map(center=[39.0, -105.5], zoom=6)
m.addLayer(colorado, {'color': 'red'}, "Colorado Boundary")
m.addLayer(points, {}, "Random Points")
m

Map(center=[39.0, -105.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [ ]:

def fc_to_dataframe(fc, n=1000):
    features = fc.limit(n).getInfo()['features']
    rows = [f['properties'] for f in features]
    return pd.DataFrame(rows)

def join_points_to_polygon_attr(
    points_fc: ee.FeatureCollection,
    polygons_fc: ee.FeatureCollection,
    poly_attrs,                     # str OR list[str]
    out_cols=None,                  # None, str, or list[str]
    keep_unmatched: bool = True,
    simplify: float = None,
    collision: str = "distinct_concat",  # "first", "concat", "distinct_concat"
    sep: str = "|"
) -> ee.FeatureCollection:

    # Normalize poly_attrs / out_cols to Python lists
    if isinstance(poly_attrs, str):
        poly_attrs = [poly_attrs]

    if out_cols is None:
        out_cols = list(poly_attrs)
    elif isinstance(out_cols, str):
        out_cols = [out_cols]

    if len(out_cols) != len(poly_attrs):
        raise ValueError("out_cols must be None, a single string, or same length as poly_attrs")

    if collision not in {"first", "concat", "distinct_concat"}:
        raise ValueError("collision must be one of: 'first', 'concat', 'distinct_concat'")

    polys = polygons_fc.filterBounds(points_fc.geometry())
    if simplify is not None:
        polys = polys.map(lambda f: f.simplify(simplify))

    # Outer join preserves points with no matches
    join = ee.Join.saveAll(matchesKey="__matched_polys", outer=keep_unmatched)
    spatial_filter = ee.Filter.intersects(leftField=".geo", rightField=".geo")
    joined = ee.FeatureCollection(join.apply(points_fc, polys, spatial_filter))

    attrs_ee = ee.List(poly_attrs)
    outcols_ee = ee.List(out_cols)

    def attach_attrs(pt):
        pt = ee.Feature(pt)

        has_matches = pt.propertyNames().contains("__matched_polys")
        matches = ee.List(ee.Algorithms.If(has_matches, pt.get("__matched_polys"), ee.List([])))

        def set_one(i, acc):
            acc = ee.Feature(acc)
            i = ee.Number(i)

            attr = ee.String(attrs_ee.get(i))
            outc = ee.String(outcols_ee.get(i))

            vals = ee.List(matches.map(lambda f: ee.Feature(f).get(attr)))

            def combined_value():
                if collision == "first":
                    return vals.get(0)

                vals_str = ee.List(vals.map(lambda v: ee.String(v)))
                if collision == "distinct_concat":
                    vals_str = vals_str.distinct()
                return vals_str.join(sep)

            val = ee.Algorithms.If(matches.size().gt(0), combined_value(), None)
            return acc.set(outc, val)

        # iterate over all requested attributes, setting each output column
        out_feat = ee.Feature(ee.List.sequence(0, attrs_ee.size().subtract(1)).iterate(set_one, pt))

        # drop join scratch field
        return out_feat.set("__matched_polys", None)

    out = joined.map(attach_attrs)

    if keep_unmatched:
        return out

    return out.filter(ee.Filter.notNull(out_cols))

# Join points to epa lvl3 to get ecoregion3 name
points_with_dats = join_points_to_polygon_attr(
    points,
    epa_lvl3,
    poly_attrs=['us_l3code','us_l3name'],
    out_cols=['us_l3code','us_l3name'],
    keep_unmatched=True
)

points_with_dats = join_points_to_polygon_attr(
    points_with_dats,
    epa_lvl4,
    poly_attrs='us_l4code',
    out_cols='us_l4code',
    keep_unmatched=True
)

points_with_dats = join_points_to_polygon_attr(
    points_with_dats,
    wumi_fires,
    poly_attrs='fireid',
    out_cols='fireid',
    keep_unmatched=True
)

df = fc_to_dataframe(points_with_dats)
print(df)

    point_id us_l3code                   us_l3name us_l4code  \
0          0        22  Arizona/New Mexico Plateau       22c   
1          1        21            Southern Rockies       21e   
2          2        20           Colorado Plateaus       20b   
3          3        21            Southern Rockies       21i   
4          4        26     Southwestern Tablelands       26e   
..       ...       ...                         ...       ...   
995      995        26     Southwestern Tablelands       26e   
996      996        25                 High Plains       25d   
997      997        21            Southern Rockies       21f   
998      998        20           Colorado Plateaus       20c   
999      999        21            Southern Rockies       21g   

                      fireid  
0                        NaN  
1    20201014_402030_1062390  
2                        NaN  
3                        NaN  
4                        NaN  
..                       ...  
995           

In [ ]:

def join_points_to_polys_wide(
    points_fc: ee.FeatureCollection,
    polygons_fc: ee.FeatureCollection,
    poly_str_nm: str,
    key_field: str,
    value_field: str,
    keep_unmatched: bool = True,
    simplify: float = None,
    collision: str = "concat", #'first', 'concat', 'distinct_concat'
    sep: str = "|",
    key_transform=None   # NEW
) -> ee.FeatureCollection:
    """
    Spatially join polygon attributes onto points in a *wide* (column-expanding) format.

    This function intersects each point in `points_fc` with polygons in `polygons_fc`, and
    creates one or more new point properties whose names are derived from a polygon "key".
    Each new property corresponds to one distinct key value (e.g., year), producing
    columns like:

        "{poly_str_nm}_{key}" -> "{value_field value(s)}"

    This is useful when you want a single point table where polygon-associated values
    are pivoted into separate columns (a "wide" table), rather than a long table with
    one row per match.

    Parameters
    ----------
    points_fc : ee.FeatureCollection
        FeatureCollection of point Features. Features must have geometries (i.e., `.geo`)
        because the join uses geometry intersection. If you previously stripped geometry,
        recreate it before calling this function.

    polygons_fc : ee.FeatureCollection
        FeatureCollection of polygon Features to intersect with points.

    poly_str_nm : str
        Prefix string used to build output column names. For each matched polygon, the
        output column name is:

            f"{poly_str_nm}_{key}"

        Example: poly_str_nm="fire" and key=2002 -> "fire_2002"

    key_field : str
        Name of the polygon property used as the default key to create column names.
        Ignored if `key_transform` is provided.

    value_field : str
        Name of the polygon property whose value is written into the output column
        (or merged into it if multiple polygons map to the same column).

    keep_unmatched : bool, default True
        If True, retain all input points, including points with no intersecting polygons.
        Unmatched points simply won't receive any wide columns.
        If False, drop points that do not intersect any polygon.

    simplify : float, optional
        If provided, simplifies polygons with `f.simplify(simplify)` prior to intersection.
        Units are meters (geodesic). This can substantially improve performance when
        polygons are complex and point counts are large, but may slightly alter boundaries.

    collision : {"first", "concat", "distinct_concat"}, default "concat"
        Strategy for handling cases where multiple polygons generate the *same* output
        column name for a given point (i.e., same key).
        - "first": keep the first value encountered; subsequent values are ignored.
        - "concat": append values using `sep` (may include duplicates).
        - "distinct_concat": append values, then de-duplicate within the column.

        Note: collisions can happen if polygons overlap spatially or if multiple polygons
        share the same key value.
"""
    polys = polygons_fc.filterBounds(points_fc.geometry())
    if simplify is not None:
        polys = polys.map(lambda f: f.simplify(simplify))

    if collision not in {"first", "concat", "distinct_concat"}:
        raise ValueError("collision must be one of: 'first', 'concat', 'distinct_concat'")

    join = ee.Join.saveAll(matchesKey="__matched_polys", outer=keep_unmatched)
    spatial_filter = ee.Filter.intersects(leftField=".geo", rightField=".geo")
    joined = ee.FeatureCollection(join.apply(points_fc, polys, spatial_filter))

    def _add_wide_cols(pt):
        pt = ee.Feature(pt)

        has_matches = pt.propertyNames().contains("__matched_polys")
        matches = ee.List(ee.Algorithms.If(has_matches, pt.get("__matched_polys"), ee.List([])))

        def _iter(f, acc):
            acc = ee.Dictionary(acc)
            f = ee.Feature(f)

            # -----------------------------
            # Key handling (new)
            # -----------------------------
            if key_transform is None:
                key_val = f.get(key_field)
            else:
                key_val = key_transform(f)

            col = ee.String(poly_str_nm).cat("_").cat(ee.String(key_val))
            val = ee.String(f.get(value_field))

            if collision == "first":
                return ee.Dictionary(ee.Algorithms.If(acc.contains(col), acc, acc.set(col, val)))

            if collision == "distinct_concat":
                prev = ee.String(acc.get(col, ""))
                merged = ee.List(
                    ee.Algorithms.If(prev.length().gt(0), prev.split(sep), ee.List([]))
                ).add(val).distinct()
                return acc.set(col, merged.join(sep))

            prev = ee.String(acc.get(col, ""))
            new_val = ee.String(ee.Algorithms.If(prev.length().gt(0), prev.cat(sep).cat(val), val))
            return acc.set(col, new_val)

        wide = ee.Dictionary(matches.iterate(_iter, ee.Dictionary({})))
        return pt.set(wide).set("__matched_polys", None)

    out = joined.map(_add_wide_cols)

    if keep_unmatched:
        return out

    return out.filter(ee.Filter.gt(out.propertyNames().size(), 0))


In [31]:

points_with_dats_wide = join_points_to_polys_wide(
    points,
    wumi_fires,
    poly_str_nm="fire",
    key_field="fire_year",
    value_field="fireid",
    key_transform=lambda f: ee.Number(f.get("fire_year")).int(),
    collision="concat",
    sep="|"
)


df = fc_to_dataframe(points_with_dats_wide)
print(df)

    point_id                fire_2020 fire_2008 fire_2016 fire_2002 fire_2013  \
0          0                      NaN       NaN       NaN       NaN       NaN   
1          1  20201014_402030_1062390       NaN       NaN       NaN       NaN   
2          2                      NaN       NaN       NaN       NaN       NaN   
3          3                      NaN       NaN       NaN       NaN       NaN   
4          4                      NaN       NaN       NaN       NaN       NaN   
..       ...                      ...       ...       ...       ...       ...   
995      995                      NaN       NaN       NaN       NaN       NaN   
996      996                      NaN       NaN       NaN       NaN       NaN   
997      997                      NaN       NaN       NaN       NaN       NaN   
998      998                      NaN       NaN       NaN       NaN       NaN   
999      999                      NaN       NaN       NaN       NaN       NaN   

    fire_2011 fire_2004 fir